In [ ]:
import SimpleITK as sitk
import numpy as np


def image_center_phys(img: sitk.Image) -> np.ndarray:
    size = np.array(img.GetSize(), dtype=float)
    spacing = np.array(img.GetSpacing(), dtype=float)
    direction = np.array(img.GetDirection(), dtype=float).reshape(3, 3)
    origin = np.array(img.GetOrigin(), dtype=float)

    # Index der Bildmitte (continuous index)
    center_index = (size - 1.0) / 2.0
    center_phys = origin + direction @ (center_index * spacing)
    return center_phys


def make_highres_reference_in_dmi_space(mag: sitk.Image, dmi: sitk.Image) -> sitk.Image:
    # High-res Eigenschaften von magnitude
    ref_size = mag.GetSize()
    ref_spacing = mag.GetSpacing()

    # Orientierung (Direction) von DMI übernehmen
    ref_direction = dmi.GetDirection()

    # Origin so wählen, dass das Zentrum des high-res Volumens auf das DMI-Zentrum fällt
    dmi_center = image_center_phys(dmi)

    direction = np.array(ref_direction, dtype=float).reshape(3, 3)
    size = np.array(ref_size, dtype=float)
    spacing = np.array(ref_spacing, dtype=float)
    center_index = (size - 1.0) / 2.0

    ref_origin = dmi_center - direction @ (center_index * spacing)

    ref = sitk.Image(ref_size, sitk.sitkFloat32)
    ref.SetSpacing(ref_spacing)
    ref.SetDirection(ref_direction)
    ref.SetOrigin(tuple(ref_origin.tolist()))
    return ref


def rigid_register_mag_to_dmi(mag: sitk.Image, dmi: sitk.Image) -> sitk.Transform:
    # Cast + optional mild smoothing helps MI stability
    fixed = sitk.Cast(dmi, sitk.sitkFloat32)
    moving = sitk.Cast(mag, sitk.sitkFloat32)

    # Initial transform around centers
    initial = sitk.CenteredTransformInitializer(
        fixed, moving,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.02)  # 2% sampling, usually enough

    reg.SetInterpolator(sitk.sitkLinear)

    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0,
        minStep=1e-4,
        numberOfIterations=300,
        gradientMagnitudeTolerance=1e-8
    )
    reg.SetOptimizerScalesFromPhysicalShift()

    # Multi-resolution pyramid
    reg.SetShrinkFactorsPerLevel([8, 4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([3, 2, 1, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    reg.SetInitialTransform(initial, inPlace=False)
    final_tx = reg.Execute(fixed, moving)
    return final_tx


def resample_to_reference(mag: sitk.Image, ref: sitk.Image, tx: sitk.Transform) -> sitk.Image:
    return sitk.Resample(
        mag, ref, tx,
        sitk.sitkBSpline,  # smoother for nice figures
        0.0, sitk.sitkFloat32
    )


if __name__ == "__main__":
    mag_path = "magnitude.nii"
    dmi_path = "water_amp_map.nii"

    mag = sitk.ReadImage(mag_path)
    dmi = sitk.ReadImage(dmi_path)

    tx = rigid_register_mag_to_dmi(mag, dmi)
    highref = make_highres_reference_in_dmi_space(mag, dmi)
    mag_highres_in_dmi = resample_to_reference(mag, highref, tx)

    sitk.WriteImage(highref, "highres_ref_in_dmi_space.nii.gz")
    sitk.WriteImage(mag_highres_in_dmi, "magnitude_highres_in_dmi.nii.gz")

    print("Wrote:")
    print(" - highres_ref_in_dmi_space.nii.gz")
    print(" - magnitude_highres_in_dmi.nii.gz")
